# Enkel Trading med Alpaca

In [1]:
# Importing
from dotenv import load_dotenv
import os
import pandas as pd

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import OrderRequest

In [2]:
# Loading env
load_dotenv()

KEY = os.getenv("key")
SECRET = os.getenv("secret")

## Placing Orders

In [3]:
trader = TradingClient(KEY, SECRET)

trader.submit_order(OrderRequest(
    symbol="AAPL",
    qty = 1,
    side = "buy",
    time_in_force = "day",
    type = "market",
))

print(trader.get_orders())

[{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
    'asset_id': UUID('b0b6dd9d-8b9b-48a9-ba46-b9d54906e415'),
    'canceled_at': None,
    'client_order_id': 'd885f857-c9d1-4d66-b804-2744b66999a3',
    'created_at': datetime.datetime(2026, 6, 16, 9, 45, 57, 388714, tzinfo=TzInfo(0)),
    'expired_at': None,
    'expires_at': datetime.datetime(2026, 6, 16, 20, 0, tzinfo=TzInfo(0)),
    'extended_hours': False,
    'failed_at': None,
    'filled_at': None,
    'filled_avg_price': None,
    'filled_qty': '0',
    'hwm': None,
    'id': UUID('13537a2a-a6b9-4fa3-b0c3-4028b9868838'),
    'legs': None,
    'limit_price': None,
    'notional': None,
    'order_class': <OrderClass.SIMPLE: 'simple'>,
    'order_type': <OrderType.MARKET: 'market'>,
    'position_intent': <PositionIntent.BUY_TO_OPEN: 'buy_to_open'>,
    'qty': '1',
    'ratio_qty': None,
    'replaced_at': None,
    'replaced_by': None,
    'replaces': None,
    'side': <OrderSide.BUY: 'buy'>,
    'status': <OrderStatus.N

### Ticker

Den unike identifikatoren som brukes for et verdipapir på en børs.

Eksempel: AAPL for Apple og TSLA for Tesla. der AAPL og TSLA er symboler (symbols)

<br>

### Ordretyper:
#### Market Order

En ordre som kjøper eller selger et verdipapir umiddelbart til beste tilgjengelige markedspris.

Eksempel: Kjøp 10 aksjer i Apple så raskt som mulig.

<br>

#### Limit Order

En ordre som kun utføres dersom markedet når en spesifisert pris eller bedre.

Eksempel: Kjøp Apple-aksjer dersom prisen faller til \$180.

<br>

#### Stop Order

En ordre som aktiveres når en forhåndsdefinert pris nås.

Eksempel: Selg en aksje dersom prisen faller under \$170.

<br>

#### Stop Loss

En stop-ordre som brukes for å begrense tap.

Eksempel: En investor kjøper en aksje til \$100 og setter stop loss på \$90.

<br>

#### Stop Limit Order

En kombinasjon av stop- og limit-ordre.

Eksempel: Når prisen faller til \$90 aktiveres ordren, men aksjen selges kun dersom prisen fortsatt er \$89 eller høyere.

<br>

#### Trailing Stop Order

En dynamisk stop-ordre som følger markedsprisen med en fast avstand.

Eksempel: En trailing stop på \$5 flyttes automatisk opp når aksjen stiger, men aldri ned.

<br>
<br>

### Time in Force (TIF)

Regler som bestemmer hvor lenge en ordre skal være aktiv.

Eksempel: En ordre kan være gyldig kun samme handelsdag eller frem til den utføres.

<br>

#### Day Order (DAY)

En ordre som kun er aktiv i inneværende handelssesjon.

Eksempel: En limit-ordre som ikke utføres før markedsstenging blir automatisk kansellert.

<br>

#### Good Till Cancelled (GTC)

En ordre som forblir aktiv til den utføres eller kanselleres manuelt.

Eksempel: En limit-ordre på \$150 kan ligge aktiv i flere uker.

<br>

#### Fill or Kill (FOK)

En ordre som må utføres umiddelbart og i sin helhet. Hvis ikke kanselleres hele ordren.

Eksempel: Kjøp 1 000 aksjer umiddelbart. Hvis alle aksjene ikke kan kjøpes med en gang, kanselleres ordren.

<br>

#### Immediate or Cancel (IOC)

En ordre som forsøker å utføre så mye som mulig umiddelbart. Resten kanselleres.

Eksempel: Kjøp 1 000 aksjer. Dersom kun 700 er tilgjengelige kjøpes disse, mens de resterende 300 kanselleres.

<br>

#### At the Open (OPG)

En ordre som sendes til markedet ved åpningen av neste handelssesjon.

Eksempel: Kjøp aksjer ved neste markedsåpning.

<br>

#### At the Close (CLS)

En ordre som forsøker å utføres så nær markedsstenging som mulig.

Eksempel: Selg en posisjon ved slutten av handelsdagen.

## Cancel Orders

In [52]:
trader.cancel_orders() # Cancel all orders
print(trader.get_orders())

#trader.cancel_order_by_id(...) #Cancel the specific order

[]


## Enkel Regel-Basert-Strategi

In [53]:
client = StockHistoricalDataClient(KEY, SECRET)

def close_bars(symbol):
    bars = client.get_stock_bars(StockBarsRequest(
        symbol_or_symbols = symbol,
        timeframe = TimeFrame.Day,
        start = "2026-01-01"
    ))

    return bars.df["close"]

def strategy(symbol):
    symbol_close_bars = close_bars(symbol)
    last_close = symbol_close_bars.iloc[-1]
    second_last_close = symbol_close_bars.iloc[-2]
    direction = last_close - second_last_close

    threshold = last_close * 0.05
    print(f"Direction = {direction}")
    if direction >= threshold:
        trader.submit_order(OrderRequest(symbol = symbol, side = "buy", qty = 1, type = "market", time_in_force = "day"))
        print("buying")
    elif direction <= -threshold:
        trader.submit_order(OrderRequest(symbol = symbol, side = "sell", qty = 1, type = "market", time_in_force = "day"))
        print("selling")
    else:
        print("holding")



In [54]:
strategy("AAPL")

Direction = 5.2900000000000205
holding
